# Tg property prompts: three variants side by side

Use the **extract** kernel. Run cells in order. Set `PAPER_UID` and `SAMPLE_ID`, then send the same paper to the three Tg prompts and compare what each returns:

1. **Per sample** (`TG_SYSTEM_PROMPT`): one request per sample; the model sees only the target record.
2. **Per paper** (`TG_PAPER_SYSTEM_PROMPT`): one request per paper (split into batches of `TARGET_BATCH_SIZE` targets); the model sees all samples, writes the evidence (`paper_label`, `identifying_data`, `match_basis`) and links each value to a sample. Code keeps only strong matches.
3. **Per sample with other samples as context** (`TG_CTX_SYSTEM_PROMPT`): like 1, but the paper's other samples are listed to tell the samples apart.

Prompts 1 and 3 use `SAMPLE_ID`, or every sample of the paper when `RUN_ALL_SAMPLES = True` (needed for a full comparison with prompt 2). Prompts are copied from `notebooks/synthesis_property_extraction.ipynb`, with one change: the `description` instruction is rewritten in all three prompts to ask for a fully specified measurement description, and prompt 2 gets a `description` field. Edits here are not carried over. Nothing is saved; the database and papers are only read.

**Sample IDs changed** with the sample-extraction re-run of October 2026, so sample IDs from older notebooks do not apply; list the samples below to pick one.

## The 20 papers

ID000061162, ID000137005, ID000143254, ID000180254, ID000180835, ID000415115, ID000573272, ID000581823, ID000588793, ID000732356, ID000732480, ID000750264, ID000752848, ID000905310, ID001002844, ID001032337, ID001098286, ID001098538, ID001098910, ID001100849.

Papers whose property-presence label includes Tg are marked by the listing cell below.

In [ ]:
import sys
sys.dont_write_bytecode = True
import asyncio
import json
import re
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

# Find the repository from this folder or any parent.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())

BASE_URL = "http://localhost:8000/v1"
MODEL = "qwen3.6-27b"
THINKING = True        # Set False for thinking off.
MAX_TOKENS = 20_000    # Completion tokens per request.
SHOW_REASONING = True # Print the model's reasoning trace above each answer.

## Prompt 1: per sample

In [ ]:
TG_SYSTEM_PROMPT = """
# Task

Extract every **glass transition temperature (Tg)** reported for **one requested polymer/material sample** from the supplied paper.

Use the database record **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tg for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a Tg only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., "Tg increased with filler content").
- Sample names can be shared by several samples, so identify the target from its whole record, especially the preparation or charecterization conditions it describes, and extract a value only when the paper reports it for a sample made under those same conditions.
- If it is unclear whether a value belongs to this sample, skip it.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same Tg measured by different methods, subtypes, heating rates, scans, or frequencies → separate records.

## 2. Fields

- **name:** always `glass_transition_temperature`.
- **method:** the technique used to measure this Tg: `DSC`, `DTA`, `DMA`, `TMA`, `dilatometry`, `dielectric`, `other`, or `unspecified` (if not stated). General instrument descriptions in the Methods section apply to this sample's measurement.
- **method_subtype:** which feature of the curve was taken as Tg. Use only if the paper states it; never infer it.
  - DSC or DTA: `onset`, `midpoint`, `inflection`, or `unspecified`.
  - DMA: `tan_delta_peak`, `loss_modulus_peak`, `storage_modulus_onset`, or `unspecified`.
  - All other methods: `null`.
  - A stated subtype not in these lists → `unspecified`, with the detail in `description`.
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit (e.g., `"152.3 ± 1.2 °C"`, `"120–125 °C"`, `"> 200 °C"`).
- **value_type** and **value:** classify the reported form and set a single number:

| value_type | Example | value |
|---|---|---|
| `exact` | `152 °C` | 152 |
| `mean_sd` | `152 ± 2 °C` | 152 |
| `approximate` | `~150 °C` | 150 |
| `range` | `120–125 °C` | 125 (upper end) |
| `lower_bound` | `> 200 °C` | 200 |
| `upper_bound` | `< −50 °C` | −50 |
| `not_observed` | `no Tg detected` | `null` |

  Use `not_observed` only when the paper explicitly states that no Tg was detected for this sample.

- **unit:** `°C`, `K`, or `°F`, as reported. Never convert.
- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has:
  - `name`: `heating_rate`, `cooling_rate`, `scan` (e.g., `first_heating`, `second_heating`, `cooling`), `frequency`, `atmosphere`, `temperature_range`, `thermal_history` (treatment of the specimen just before measurement, e.g., quenching), `specimen_form` (e.g., film, powder), or `other`.
  - `type`: `exact`, `mean_sd`, `range`, `approximate`, `lower_bound`, or `upper_bound`.
  - `value`: a number when numeric (e.g., `10`), otherwise text (e.g., `"N2"`, `"second_heating"`).
  - `unit`: e.g., `"K/min"`, `"Hz"`, `"°C"`; `null` for non-numeric values.
  - `description`: `null`, or a short note (required when `name` is `other`).

  Include only conditions the paper states. Use an empty list if none.
- **origin:** `measured`, or `calculated` if the value comes from a model or equation (e.g., Fox, Gordon–Taylor, simulation).
- **description:** a clean, fully specified description of this measurement that can be understood without the paper or the other fields. Include everything the paper states about it, even if it is already in another field:
  - what the value is: which transition, phase, segment, or block (e.g., soft-segment Tg, second Tg of a blend);
  - the method and how Tg was taken from the curve (e.g., midpoint of the heat-capacity step, tan δ peak);
  - the scan and thermal history (e.g., second heating after cooling at 10 °C/min; annealing or quenching before the test);
  - heating or cooling rate, frequency, strain or amplitude, temperature range, atmosphere, instrument, and specimen form, mass, or preparation for the test;
  - any other measurement detail not captured in the fields.

  Do not include the value, the sample name or identifier, table or figure numbers, or comparisons with other samples. Use only what the paper states; if little is stated, say so briefly (e.g., "Tg measured by DSC; no measurement conditions stated.").

  Example: "Tg of the soft polyether segment, taken as the midpoint of the heat-capacity step in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −90 to 200 °C after cooling at 10 °C/min; about 5 mg of film in an aluminium pan."

## 3. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. Keep each value with its correct method and conditions. If a detail is not stated, use `unspecified`, `null`, or an empty list.

## 4. Output

Before returning, verify that:

- Every record is for the requested sample.
- No cited, plotted-only, or other-sample values are included.
- `value` follows the table above.
- `method_subtype` matches `method`.
- `description` is fully specified and contains no value, sample name, or table or figure number.

Return **only** the JSON object. No explanations or extra fields.
"""

TG_USER_PROMPT = """
Extract every glass transition temperature reported for the requested sample, following the system instructions.

Return exactly `{"properties": [...]}`, or `{"properties": []}` if none is reported for this sample.

Use the identification record only to identify the target material. Extract all values and measurement details exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

TG_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "sample_tg",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["properties"],
            "properties": {
                "properties": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": [
                            "name", "method", "method_subtype", "value_raw", "value",
                            "value_type", "unit", "conditions", "origin", "description"
                        ],
                        "properties": {
                            "name": {"type": "string", "enum": ["glass_transition_temperature"]},
                            "method": {
                                "type": "string",
                                "enum": ["DSC", "DTA", "DMA", "TMA", "dilatometry", "dielectric", "other", "unspecified"]
                            },
                            "method_subtype": {
                                "type": ["string", "null"],
                                "enum": [
                                    "onset", "midpoint", "inflection",
                                    "tan_delta_peak", "loss_modulus_peak", "storage_modulus_onset",
                                    "unspecified", None
                                ]
                            },
                            "value_raw": {"type": "string"},
                            "value": {"type": ["number", "null"]},
                            "value_type": {
                                "type": "string",
                                "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound", "not_observed"]
                            },
                            "unit": {"type": "string", "enum": ["°C", "K", "°F"]},
                            "conditions": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "additionalProperties": False,
                                    "required": ["name", "type", "value", "unit", "description"],
                                    "properties": {
                                        "name": {
                                            "type": "string",
                                            "enum": [
                                                "heating_rate", "cooling_rate", "scan", "frequency", "atmosphere",
                                                "temperature_range", "thermal_history", "specimen_form", "other"
                                            ]
                                        },
                                        "type": {
                                            "type": "string",
                                            "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound"]
                                        },
                                        "value": {"type": ["number", "string"]},
                                        "unit": {"type": ["string", "null"]},
                                        "description": {"type": ["string", "null"]}
                                    }
                                }
                            },
                            "origin": {"type": "string", "enum": ["measured", "calculated"]},
                            "description": {"type": "string"}
                        }
                    }
                }
            }
        }
    }
}

## Prompt 2: per paper

The response format is built per paper so `sample_id` can only be one of that paper's IDs (or null). Linking fields come first so the model writes the evidence before it picks the sample.

Compared with the original per-paper prompt, this copy adds a `description` field (after `origin`), with the same instruction as prompts 1 and 3.

In [ ]:
TG_PAPER_SYSTEM_PROMPT = """
# Task

Extract every **glass transition temperature (Tg)** value reported in the supplied paper and link each value to a sample from the supplied sample list.

The sample list contains every sample recorded for this paper. Use it **only to identify samples**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tg for the target samples, return `{"properties": []}`.

## 1. Which values to report

Report every Tg value the paper gives for a target sample, and every Tg value that matches no sample in the list. Skip values that clearly belong to a listed sample that is not a target.

- Do not report values quoted from other publications.
- Do not read values off plots. Use only numbers stated in text, tables, or captions.
- Do not report trends without a number.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same Tg measured by different methods, subtypes, heating rates, scans, or frequencies → separate records.
- If the paper states that one value applies to several listed samples, report one record per sample.

## 2. Linking a value to a sample

For each value, first record what the paper says about the material, then choose the sample.

- **paper_label:** the name or label the paper uses for the material next to the value (e.g., the table row label).
- **identifying_data:** short text with the data reported together with the value that identify the material: table entry, preparation conditions, composition, molecular weight, dispersity. Use `""` if none.
- **match_basis:** how the value is tied to the chosen sample:
  - `same_entry`: the paper refers the value to the same table entry, run, or batch as the sample record.
  - `same_conditions`: the preparation conditions given with the value agree with the sample record.
  - `same_characterization`: the characterization data given with the value (molecular weight, dispersity, composition) agree with the sample record.
  - `name_only`: only the name or label matches; nothing else ties the value to this sample.
  - `none`: no listed sample matches.
- **sample_id:** the `sample_id` of the matched sample, or `null` when `match_basis` is `none`.

Rules:

- Several samples can share a name. A shared name alone never decides between them; compare the identifying data with each record.
- If the identifying data disagree with a sample record (e.g., a different molecular weight or different preparation conditions), the value does not belong to that sample.
- If no listed sample agrees with the identifying data, use `match_basis: none` and `sample_id: null`. This is a correct and common answer.

## 3. Value fields

- **name:** always `glass_transition_temperature`.
- **method:** the technique used to measure this Tg: `DSC`, `DTA`, `DMA`, `TMA`, `dilatometry`, `dielectric`, `other`, or `unspecified` (if not stated). Table footnotes and general instrument descriptions in the Methods section count as stated.
- **method_subtype:** which feature of the curve was taken as Tg. Use only if the paper states it; never infer it.
  - DSC or DTA: `onset`, `midpoint`, `inflection`, or `unspecified`.
  - DMA: `tan_delta_peak`, `loss_modulus_peak`, `storage_modulus_onset`, or `unspecified`.
  - All other methods: `null`.
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit.
- **value_type** and **value:**

| value_type | Example | value |
|---|---|---|
| `exact` | `152 °C` | 152 |
| `mean_sd` | `152 ± 2 °C` | 152 |
| `approximate` | `~150 °C` | 150 |
| `range` | `120–125 °C` | 125 (upper end) |
| `lower_bound` | `> 200 °C` | 200 |
| `upper_bound` | `< −50 °C` | −50 |
| `not_observed` | `no Tg detected` | `null` |

  Use `not_observed` only when the paper explicitly states that no Tg was detected.

- **unit:** `°C`, `K`, or `°F`, as reported. Never convert.
- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has `name` (`heating_rate`, `cooling_rate`, `scan`, `frequency`, `atmosphere`, `temperature_range`, `thermal_history`, `specimen_form`, or `other`), `type`, `value` (number when numeric, otherwise text), `unit` (`null` for non-numeric values), and `description` (`null`, or a short note; required for `other`). Use an empty list if none are stated.
- **origin:** `measured`, or `calculated` if the value comes from a model or equation.
- **description:** a clean, fully specified description of this measurement that can be understood without the paper or the other fields. Include everything the paper states about it, even if it is already in another field:
  - what the value is: which transition, phase, segment, or block (e.g., soft-segment Tg, second Tg of a blend);
  - the method and how Tg was taken from the curve (e.g., midpoint of the heat-capacity step, tan δ peak);
  - the scan and thermal history (e.g., second heating after cooling at 10 °C/min; annealing or quenching before the test);
  - heating or cooling rate, frequency, strain or amplitude, temperature range, atmosphere, instrument, and specimen form, mass, or preparation for the test;
  - any other measurement detail not captured in the fields.

  Do not include the value, the sample name or identifier, table or figure numbers, or comparisons with other samples. Use only what the paper states; if little is stated, say so briefly (e.g., "Tg measured by DSC; no measurement conditions stated.").

  Example: "Tg of the soft polyether segment, taken as the midpoint of the heat-capacity step in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −90 to 200 °C after cooling at 10 °C/min; about 5 mg of film in an aluminium pan."

## 4. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. If a detail is not stated, use `unspecified`, `null`, `""`, or an empty list.

Before returning, verify that every `description` is fully specified and contains no value, sample name, or table or figure number.

Return **only** the JSON object. No explanations or extra fields.
"""

TG_PAPER_USER_PROMPT = """
Extract every glass transition temperature reported in the paper and link each value to a sample from the sample list, following the system instructions.

Return exactly `{"properties": [...]}`.

paper_uid={paper_uid}
Target samples: {target_sample_ids}

<SAMPLE_LIST_DATA_ONLY>
{sample_list}
</SAMPLE_LIST_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""


def make_tg_paper_response_format(sample_ids):
    # Built per paper so sample_id can only be one of the paper's sample IDs (or null).
    # Linking fields come first so the model writes the evidence before it picks the sample.
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "paper_tg",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "required": ["properties"],
                "properties": {
                    "properties": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": [
                                "paper_label", "value_raw", "identifying_data", "match_basis", "sample_id",
                                "name", "method", "method_subtype", "value_type", "value", "unit",
                                "conditions", "origin", "description",
                            ],
                            "properties": {
                                "paper_label": {"type": "string"},
                                "value_raw": {"type": "string"},
                                "identifying_data": {"type": "string"},
                                "match_basis": {
                                    "type": "string",
                                    "enum": ["same_entry", "same_conditions", "same_characterization", "name_only", "none"],
                                },
                                "sample_id": {"type": ["integer", "null"], "enum": [int(i) for i in sample_ids] + [None]},
                                "name": {"type": "string", "enum": ["glass_transition_temperature"]},
                                "method": {
                                    "type": "string",
                                    "enum": ["DSC", "DTA", "DMA", "TMA", "dilatometry", "dielectric", "other", "unspecified"],
                                },
                                "method_subtype": {
                                    "type": ["string", "null"],
                                    "enum": ["onset", "midpoint", "inflection", "tan_delta_peak", "loss_modulus_peak",
                                             "storage_modulus_onset", "unspecified", None],
                                },
                                "value_type": {
                                    "type": "string",
                                    "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound", "not_observed"],
                                },
                                "value": {"type": ["number", "null"]},
                                "unit": {"type": "string", "enum": ["°C", "K", "°F"]},
                                "conditions": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "additionalProperties": False,
                                        "required": ["name", "type", "value", "unit", "description"],
                                        "properties": {
                                            "name": {
                                                "type": "string",
                                                "enum": ["heating_rate", "cooling_rate", "scan", "frequency", "atmosphere",
                                                         "temperature_range", "thermal_history", "specimen_form", "other"],
                                            },
                                            "type": {
                                                "type": "string",
                                                "enum": ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound"],
                                            },
                                            "value": {"type": ["number", "string"]},
                                            "unit": {"type": ["string", "null"]},
                                            "description": {"type": ["string", "null"]},
                                        },
                                    },
                                },
                                "origin": {"type": "string", "enum": ["measured", "calculated"]},
                                "description": {"type": "string"},
                            },
                        },
                    },
                },
            },
        },
    }

## Prompt 3: per sample, other samples as context

Same output format as prompt 1 (`TG_RESPONSE_FORMAT`), so the results can be compared directly.

In [ ]:
TG_CTX_SYSTEM_PROMPT = """
# Task

Extract every **glass transition temperature (Tg)** reported for **one requested polymer/material sample** from the supplied paper.

Use the target record and the list of the paper's other samples **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tg for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a Tg only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., "Tg increased with filler content").
- Sample names can be shared by several samples, so identify the target from its whole record, especially the preparation or charecterization conditions it describes, and extract a value only when the paper reports it for a sample made under those same conditions.
- The paper's other samples are listed only so you can tell the samples apart. Compare the target record with them, and extract a value only if the paper reports it for the target and not for one of the other listed samples or for a material that is not in the list. Never extract values for the other samples.
- If it is unclear whether a value belongs to this sample, skip it.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same Tg measured by different methods, subtypes, heating rates, scans, or frequencies → separate records.

## 2. Fields

- **name:** always `glass_transition_temperature`.
- **method:** the technique used to measure this Tg: `DSC`, `DTA`, `DMA`, `TMA`, `dilatometry`, `dielectric`, `other`, or `unspecified` (if not stated). General instrument descriptions in the Methods section apply to this sample's measurement.
- **method_subtype:** which feature of the curve was taken as Tg. Use only if the paper states it; never infer it.
  - DSC or DTA: `onset`, `midpoint`, `inflection`, or `unspecified`.
  - DMA: `tan_delta_peak`, `loss_modulus_peak`, `storage_modulus_onset`, or `unspecified`.
  - All other methods: `null`.
  - A stated subtype not in these lists → `unspecified`, with the detail in `description`.
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit (e.g., `"152.3 ± 1.2 °C"`, `"120–125 °C"`, `"> 200 °C"`).
- **value_type** and **value:** classify the reported form and set a single number:

| value_type | Example | value |
|---|---|---|
| `exact` | `152 °C` | 152 |
| `mean_sd` | `152 ± 2 °C` | 152 |
| `approximate` | `~150 °C` | 150 |
| `range` | `120–125 °C` | 125 (upper end) |
| `lower_bound` | `> 200 °C` | 200 |
| `upper_bound` | `< −50 °C` | −50 |
| `not_observed` | `no Tg detected` | `null` |

  Use `not_observed` only when the paper explicitly states that no Tg was detected for this sample.

- **unit:** `°C`, `K`, or `°F`, as reported. Never convert.
- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has:
  - `name`: `heating_rate`, `cooling_rate`, `scan` (e.g., `first_heating`, `second_heating`, `cooling`), `frequency`, `atmosphere`, `temperature_range`, `thermal_history` (treatment of the specimen just before measurement, e.g., quenching), `specimen_form` (e.g., film, powder), or `other`.
  - `type`: `exact`, `mean_sd`, `range`, `approximate`, `lower_bound`, or `upper_bound`.
  - `value`: a number when numeric (e.g., `10`), otherwise text (e.g., `"N2"`, `"second_heating"`).
  - `unit`: e.g., `"K/min"`, `"Hz"`, `"°C"`; `null` for non-numeric values.
  - `description`: `null`, or a short note (required when `name` is `other`).

  Include only conditions the paper states. Use an empty list if none.
- **origin:** `measured`, or `calculated` if the value comes from a model or equation (e.g., Fox, Gordon–Taylor, simulation).
- **description:** a clean, fully specified description of this measurement that can be understood without the paper or the other fields. Include everything the paper states about it, even if it is already in another field:
  - what the value is: which transition, phase, segment, or block (e.g., soft-segment Tg, second Tg of a blend);
  - the method and how Tg was taken from the curve (e.g., midpoint of the heat-capacity step, tan δ peak);
  - the scan and thermal history (e.g., second heating after cooling at 10 °C/min; annealing or quenching before the test);
  - heating or cooling rate, frequency, strain or amplitude, temperature range, atmosphere, instrument, and specimen form, mass, or preparation for the test;
  - any other measurement detail not captured in the fields.

  Do not include the value, the sample name or identifier, table or figure numbers, or comparisons with other samples. Use only what the paper states; if little is stated, say so briefly (e.g., "Tg measured by DSC; no measurement conditions stated.").

  Example: "Tg of the soft polyether segment, taken as the midpoint of the heat-capacity step in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −90 to 200 °C after cooling at 10 °C/min; about 5 mg of film in an aluminium pan."

## 3. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. Keep each value with its correct method and conditions. If a detail is not stated, use `unspecified`, `null`, or an empty list.

## 4. Output

Before returning, verify that:

- Every record is for the requested sample.
- No cited, plotted-only, or other-sample values are included.
- `value` follows the table above.
- `method_subtype` matches `method`.
- `description` is fully specified and contains no value, sample name, or table or figure number.

Return **only** the JSON object. No explanations or extra fields.
"""

TG_CTX_USER_PROMPT = """
Extract every glass transition temperature reported for the requested sample, following the system instructions.

Return exactly `{"properties": [...]}`, or `{"properties": []}` if none is reported for this sample.

Use the target record and the other samples only to identify the target material. Extract all values and measurement details exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

Other samples in this paper, for telling samples apart only (do not extract values for them):

<OTHER_SAMPLES_DATA_ONLY>
{other_samples}
</OTHER_SAMPLES_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

# Output format is the same as the per-sample prompt: TG_RESPONSE_FORMAT.

## Helpers

Database access, response parsing, and the checks that strict JSON schema cannot express (`value` is null exactly for `not_observed`; `method_subtype` must fit `method`).

In [ ]:
TG_SUBTYPES = {
    "DSC": {"onset", "midpoint", "inflection", "unspecified"},
    "DTA": {"onset", "midpoint", "inflection", "unspecified"},
    "DMA": {"tan_delta_peak", "loss_modulus_peak", "storage_modulus_onset", "unspecified"},
}


def check_tg_record(record):
    problems = []
    if (record.get("value") is None) != (record.get("value_type") == "not_observed"):
        problems.append("value must be null exactly when value_type is not_observed")
    allowed = TG_SUBTYPES.get(record.get("method"))
    subtype = record.get("method_subtype")
    if allowed is None and subtype is not None:
        problems.append(f"method_subtype must be null for method {record.get('method')}")
    if allowed is not None and subtype not in allowed:
        problems.append(f"method_subtype {subtype!r} not allowed for {record.get('method')}")
    return "; ".join(problems)


def format_conditions(conditions):
    parts = []
    for condition in conditions or []:
        text = f"{condition.get('name')}={condition.get('value')}"
        if condition.get("unit"):
            text += f" {condition['unit']}"
        if condition.get("type") not in (None, "exact"):
            text += f" ({condition['type']})"
        parts.append(text)
    return "; ".join(parts)


async def ask_qwen(client, messages, response_format):
    return await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
        extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
        response_format=response_format,
    )


def get_reasoning(response):
    # vLLM returns the thinking trace as `reasoning_content` (older) or `reasoning` (newer).
    if isinstance(response, Exception):
        return ""
    message = response.choices[0].message
    reasoning = getattr(message, "reasoning_content", None) or getattr(message, "reasoning", None)
    if not reasoning and getattr(message, "model_extra", None):
        reasoning = message.model_extra.get("reasoning_content") or message.model_extra.get("reasoning")
    if not reasoning:
        match = re.search(r"<think>(.*?)</think>", message.content or "", flags=re.S)
        reasoning = match.group(1) if match else ""
    return (reasoning or "").strip()


def print_reasoning(response):
    if not SHOW_REASONING:
        return
    print("--- Reasoning ---")
    print(get_reasoning(response) or "[No reasoning returned; is THINKING on?]")
    print("--- Answer ---")


def parse_response(response):
    # Returns (parsed JSON or None, note); note is empty when everything went fine.
    if isinstance(response, Exception):
        return None, f"request failed: {response!r}"
    notes = []
    if response.choices[0].finish_reason != "stop":
        notes.append(f"Response may be incomplete. Finish reason: {response.choices[0].finish_reason}")
    answer = response.choices[0].message.content or ""
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()
    try:
        return json.loads(answer), "\n".join(notes)
    except json.JSONDecodeError:
        notes.append(answer or "[No answer content returned]")
        return None, "\n".join(notes)


def read_db(query, params):
    connection = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)
    try:
        return pd.read_sql_query(query, connection, params=params)
    finally:
        connection.close()


def get_samples(paper_uid):
    return read_db(
        "SELECT s.paper_uid, p.canonical_source AS source, s.sample_id, "
        "s.name, s.identifier, s.aliases, s.description "
        "FROM samples s JOIN papers p USING (paper_uid) "
        "WHERE s.paper_uid = ? ORDER BY s.sample_id",
        (paper_uid,),
    )


def get_paper_title(paper_uid):
    rows = read_db("SELECT title FROM papers WHERE paper_uid = ?", (paper_uid,))
    return rows.title.iloc[0] if not rows.empty and rows.title.iloc[0] else "[no title in database]"


def get_property_presence(paper_uid):
    rows = read_db("SELECT status, properties_json FROM property_classification WHERE paper_uid = ?",
                   (paper_uid,))
    if rows.empty:
        return "not classified", []
    if rows.status.iloc[0] != "success":
        return rows.status.iloc[0], []
    return "success", [name.strip().lower() for name in json.loads(rows.properties_json.iloc[0])]


def sample_json(r):
    return json.dumps({
        "sample_id": int(r.sample_id), "name": r.name, "identifier": r.identifier,
        "aliases": json.loads(r.aliases), "description": r.description,
    }, ensure_ascii=False)


def tg_label(r):
    # Short text for one Tg record, used in the comparison tables.
    subtype = r.get("method_subtype")
    method = r.get("method") or ""
    if subtype not in (None, "unspecified"):
        method += f"/{subtype}"
    conditions = format_conditions(r.get("conditions"))
    return f"{r.get('value_raw')} [{method}{'; ' + conditions if conditions else ''}]"

## Choose a paper and a sample

Set `PAPER_UID`, run the listing cell, then set `SAMPLE_ID` to one of the listed IDs. `RUN_ALL_SAMPLES = True` runs prompts 1 and 3 for every sample of the paper (one request per sample) so all three prompts cover the same samples.

In [18]:
PAPER_UID = "ID000588793"
SAMPLE_ID = 1751766          # set to a sample_id from the list below
RUN_ALL_SAMPLES = True   # True: prompts 1 and 3 run every sample of the paper
MAX_CONCURRENT = 4        # parallel requests for prompts 1 and 3
TARGET_BATCH_SIZE = 15    # prompt 2: target samples per request

In [19]:
# Paper UID	Samples	Title (short)
# ID000061162	4	PLA / capsicum-oleoresin silica composites
# ID000143254	4	Waterborne acrylic copolymer resin for inks
# ID000415115	6	Chemically modified lignin addition
# ID000573272	5	Glycolysis of waste PET via reactive extrusion
# ID000581823	8	PS-b-P4VP thermal properties, rheology and foams
# ID000588793	13	Structure–property relationships, elasticity
# ID000732356	29	Ring-opening polymerisation of oxoester/thioester dimers
# ID000732480	6	PLA with thionoester linkages
# ID000750264	2	Surface-grafted thermoresponsive hydrogel
# ID000752848	5	Cellulose nanocrystal trans-crystallisation
# ID000905310	25	Natural-fibre biopolymer composites (the paper flagged as unreliable)
# ID001002844	20	Amphiphilic poly(ethylene phosphate) block copolymers
# ID001032337	6	Biobased thermoplastic elastomers
# ID001098286	6	Interfacial entropic interactions, fragility
# ID001098538	4	Magnetic shape-memory polymers
# ID001098910	7	PCL / graphite nanoplate nanopapers

In [20]:
paper_samples = get_samples(PAPER_UID)
if paper_samples.empty:
    raise ValueError(f"No samples for {PAPER_UID}.")
md_path = ROOT / "data/fulltext" / paper_samples.source.iloc[0] / PAPER_UID / f"{PAPER_UID}.md"
paper_markdown = md_path.read_text(encoding="utf-8")
if not paper_markdown.strip():
    raise ValueError(f"The paper Markdown is empty: {md_path}")

status, present = get_property_presence(PAPER_UID)
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(paper_samples)} samples; full text {len(paper_markdown):,} characters")
print(f"Property presence ({status}): {', '.join(present) or 'none'}")
print("Tg flagged for this paper:", "glass transition temperature" in present)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(paper_samples[["sample_id", "name", "identifier", "aliases", "description"]])

if RUN_ALL_SAMPLES:
    run_ids = [int(i) for i in paper_samples.sample_id]
else:
    if SAMPLE_ID is None or int(SAMPLE_ID) not in set(paper_samples.sample_id.astype(int)):
        raise ValueError(f"Set SAMPLE_ID to one of the sample_ids above (or RUN_ALL_SAMPLES = True).")
    run_ids = [int(SAMPLE_ID)]
print(f"Prompts 1 and 3 will run for: {run_ids}")

ID000588793: Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers
13 samples; full text 32,945 characters
Property presence (success): number-average molecular weight, weight-average molecular weight, dispersity, degree of polymerization, glass transition temperature, melting temperature, thermal decomposition temperature, young's modulus, tensile strength, elongation at break, storage modulus, degree of crystallinity, d-spacing
Tg flagged for this paper: True


,sample_id,name,identifier,aliases,description
0,312469,(PEB-b-PTMO-b-PEB)n mBCP,P1-55K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 1:1 hard-to-soft segment mass ratio. Synthesized via cascade polycondensation-coupled rapid ring-opening polymerization (PROP) of ethylene brassylate (EB) and dihydroxyl-terminated PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 57.2 kDa, a PTMO weight fraction of 59.8%, and a block number (n) of 12."
1,312470,(PEB-b-PTMO-b-PEB)n mBCP,P1-80K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 1:1 hard-to-soft segment mass ratio. Synthesized via cascade PROP of EB and PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 62.2 kDa, a PTMO weight fraction of 65.4%, and a block number (n) of 14."
2,312471,(PEB-b-PTMO-b-PEB)n mBCP,P1-94K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 1:1 hard-to-soft segment mass ratio. Synthesized via cascade PROP of EB and PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 108 kDa, a PTMO weight fraction of 66.8%, and a block number (n) of 25."
3,312472,(PEB-b-PTMO-b-PEB)n mBCP,P1-122K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 1:1 hard-to-soft segment mass ratio. Synthesized via cascade PROP of EB and PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 129 kDa, a PTMO weight fraction of 59.8%, and a block number (n) of 27."
4,312473,(PEB-b-PTMO-b-PEB)n mBCP,P2-89K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 2:1 hard-to-soft segment mass ratio. Synthesized via cascade PROP of EB and PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 83.6 kDa, a PTMO weight fraction of 39.9%, and a block number (n) of 10."
5,312474,(PEB-b-PTMO-b-PEB)n mBCP,P3-96K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 3:1 hard-to-soft segment mass ratio. Synthesized via cascade PROP of EB and PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 85.8 kDa, a PTMO weight fraction of 34.3%, and a block number (n) of 11."
6,312475,(PEB-b-PTMO-b-PEB)n mBCP,P4-96K,[],"Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymer with a 4:1 hard-to-soft segment mass ratio. Synthesized via cascade PROP of EB and PTMO (Mn = 2900 g/mol) using Ti(n-C4H9O)4 (0.05 wt%) as catalyst, followed by chloroform dissolution, methanol precipitation, and vacuum drying. Distinguished by an NMR-determined Mn of 87.1 kDa, a PTMO weight fraction of 28.1%, and a block number (n) of 9."
7,312476,P1-94K melt-spun fiber,P1-94K-OF,"[""original fiber""]","Elastic fiber prepared by melt-spinning dried P1-94K elastomer powder through a 1 mm single-hole spinneret at 110 °C with a winding speed of 2.5 m/min. Distinguished by being the as-spun, undrawn fiber form of P1-94K, exhibiting an elastic recovery of 79% and pronounced hysteresis due to disordered segment arrangement."
8,312477,P1-94K post-drawn fiber,P1-94K-SF-2,[],"Elastic fiber prepared by melt-spinning P1-94K followed by room temperature post-drawing at a draw ratio of 2. Distinguished by exhibitin

Prompts 1 and 3 will run for: [312469, 312470, 312471, 312472, 312473, 312474, 312475, 312476, 312477, 312478, 312479, 312480, 312481]


## Send prompt 1 (per sample)

One request per sample in `run_ids`. Records are in `p1_records`; reasoning in `p1_reasoning`.

In [21]:
def build_p1_messages(sample_id):
    t = paper_samples.loc[paper_samples.sample_id == sample_id].iloc[0]
    values = {
        "paper_uid": PAPER_UID,
        "sample_id": str(sample_id),
        "sample_record": json.dumps({k: t[k] for k in ("name", "identifier", "aliases", "description")},
                                    ensure_ascii=False),
        "markdown_full_text": paper_markdown,
    }
    user = re.sub(r"\{(paper_uid|sample_id|sample_record|markdown_full_text)\}",
                  lambda m: values[m.group(1)], TG_USER_PROMPT)
    return [{"role": "system", "content": TG_SYSTEM_PROMPT}, {"role": "user", "content": user}]


semaphore = asyncio.Semaphore(MAX_CONCURRENT)
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    async def run_one(sample_id):
        async with semaphore:
            return await ask_qwen(client, build_p1_messages(sample_id), TG_RESPONSE_FORMAT)
    p1_responses = dict(zip(run_ids, await asyncio.gather(*(run_one(i) for i in run_ids),
                                                          return_exceptions=True)))
p1_reasoning = {i: get_reasoning(r) for i, r in p1_responses.items()}

names = paper_samples.set_index("sample_id")
rows = []
for sample_id, response in p1_responses.items():
    print(f"\n=== {sample_id}: {names['name'][sample_id]} | {names['identifier'][sample_id]} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    if records is None:
        print(note or "[No parsable answer]")
        continue
    if not records:
        print("No values reported for this sample.")
    for r in records:
        rows.append({
            "sample_id": sample_id, "label": tg_label(r),
            "method": r.get("method"), "method_subtype": r.get("method_subtype"),
            "value_raw": r.get("value_raw"), "value": r.get("value"), "value_type": r.get("value_type"),
            "unit": r.get("unit"), "origin": r.get("origin"),
            "conditions": format_conditions(r.get("conditions")),
            "description": r.get("description"), "check": check_tg_record(r),
        })
    if note:
        print(note)

p1_records = pd.DataFrame(rows)
if not p1_records.empty:
    dup = p1_records.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    p1_records.loc[dup, "check"] = (p1_records.loc[dup, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nPrompt 1 Tg values ({len(p1_records)}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(p1_records.drop(columns="label"))


=== 312469: (PEB-b-PTMO-b-PEB)n mBCP | P1-55K ===
--- Reasoning ---
The user wants to extract the glass transition temperature (Tg) for the sample identified as "P1-55K" (or "(PEB-b-PTMO-b-PEB)n mBCP" with identifier "P1-55K").
I need to scan the paper for Tg values reported for this specific sample.
The paper discusses a series of mBCPs: P1-55K, P1-80K, P1-94K, P1-122K, P2-89K, P3-96K, P4-96K.
Table 1 lists molecular characteristics.
Table S2 and S3 are mentioned for thermal parameters, but I only have the main text.
Let's search for "Tg" or "glass transition" in the text.
Text says: "Due to the rapid crystallization of PEB, its glass transition (Tg) was not detected by DSC (Fig. 2 b) [41]; By magnifying the DSC curve in Fig. 2 b over the range of −20 to 20 °C (Fig. S5), a weak glass transition corresponding to the PEB segment becomes evident (−4 to 2 °C)."
This is a general statement for the mBCPs, not specific to P1-55K.
Text says: "Loss modulus and tan δ revealed relaxation peaks 

,sample_id,method,method_subtype,value_raw,value,value_type,unit,origin,conditions,description,check
0,312470,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,heating_rate=10 °C/min; scan=second_heating; atmosphere=N2,"Glass transition temperature of the PEB hard segment, observed as a weak transition in the second DSC heating scan recorded at 10 °C/min under nitrogen atmosphere.",assigned to several samples
1,312470,DMA,tan_delta_peak,−40 to −50 °C,-40,range,°C,measured,frequency=1 Hz; heating_rate=3 °C/min,"Glass transition temperature of the PTMO soft segment, identified from the relaxation peaks in the loss modulus and tan δ curves measured by DMA at 1 Hz with a heating rate of 3 °C/min.",
2,312471,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; temperature_range=-20 to 100 °C (range); atmosphere=N2; specimen_form=film,"Tg of the PEB segment, observed as a weak transition in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −20 to 100 °C after cooling at 10 °C/min; approximately 5 mg of sample sealed in an alumina pan.",
3,312471,DMA,tan_delta_peak,−40 to −50 °C,-40,range,°C,measured,heating_rate=3 °C/min; frequency=1 Hz; temperature_range=-100 to 60 °C (range); specimen_form=dumbbell-shaped film,"Tg of the PTMO segment, identified from the tan δ and loss modulus relaxation peaks in DMA tensile mode, measured at 1 Hz with a heating rate of 3 °C/min over a temperature range of −100 to 60 °C; dumbbell-shaped samples (2 mm thick, 4 mm wide) prepared by hot pressing.",
4,312474,DMA,tan_delta_peak,−40 to −50 °C,-40,range,°C,measured,frequency=1 Hz; heating_rate=3 °C/min; temperature_range=60 °C (range),"Glass transition temperature of the PTMO soft segment, identified as the relaxation peak in the loss modulus and tan δ curves measured by DMA at 1 Hz with a heating rate of 3 °C/min over a temperature range of −100 to 60 °C.",
5,312474,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,heating_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=100 °C (range),"Glass transition temperature of the PEB hard segment, observed as a weak transition in the second DSC heating scan recorded at 10 °C/min under nitrogen from −20 to 100 °C after thermal history removal.",
6,312475,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,heating_rate=10 °C/min; scan=second_heating; atmosphere=N2,"Tg of the PEB hard segment, observed as a weak transition in the second DSC heating scan recorded at 10 °C/min under nitrogen from −20 to 100 °C after an initial heating and cooling cycle to erase thermal history.",assigned to several samples
7,312475,DMA,tan_delta_peak,−40 to −50 °C,-40,range,°C,measured,frequency=1 Hz; heating_rate=3 °C/min; temperature_range=-100 to 60 °C; specimen_form=dumbbell-shaped sample,"Tg of the PTMO soft segment, identified as the relaxation peak in the tan δ and loss modulus curves measured by DMA in tensile mode at 1 Hz with a heating rate of 3 °C/min from −100 to 60 °C on hot-pressed dumbbell-shaped samples.",


## Send prompt 2 (per paper)

Needs only `PAPER_UID`. Every request lists all samples; papers with more than `TARGET_BATCH_SIZE` samples are split into requests that differ only in their targets.

Code decides which values to keep: `sample_id` set and `match_basis` is `same_entry`, `same_conditions` or `same_characterization`; `name_only` is kept only when no other sample in the paper has the same name. All values, kept or not, are in `paper_tg_records`.

In [22]:
# TARGET_BATCH_SIZE is set in the paper/sample cell above.
STRONG_MATCHES = {"same_entry", "same_conditions", "same_characterization"}

paper_samples_all = get_samples(PAPER_UID)
if paper_samples_all.empty:
    raise ValueError(f"No samples for {PAPER_UID}.")

all_ids = [int(i) for i in paper_samples_all.sample_id]
sample_list = "\n".join(
    json.dumps({
        "sample_id": int(r.sample_id), "name": r.name, "identifier": r.identifier,
        "aliases": json.loads(r.aliases), "description": r.description,
    }, ensure_ascii=False)
    for r in paper_samples_all.itertuples()
)
batches = [all_ids[i:i + TARGET_BATCH_SIZE] for i in range(0, len(all_ids), TARGET_BATCH_SIZE)]
tg_paper_format = make_tg_paper_response_format(all_ids)


def build_paper_messages(targets):
    values = {
        "paper_uid": PAPER_UID,
        "target_sample_ids": "all samples in the list" if len(batches) == 1 else ", ".join(map(str, targets)),
        "sample_list": sample_list,
        "markdown_full_text": paper_markdown,
    }
    user = re.sub(r"\{(paper_uid|target_sample_ids|sample_list|markdown_full_text)\}",
                  lambda m: values[m.group(1)], TG_PAPER_USER_PROMPT)
    return [{"role": "system", "content": TG_PAPER_SYSTEM_PROMPT}, {"role": "user", "content": user}]


print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(all_ids)} samples, {len(batches)} request(s)")

# Messages are built here, from the current prompt cell, so prompt edits always take effect.
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    paper_tg_responses = await asyncio.gather(
        *(ask_qwen(client, build_paper_messages(targets), tg_paper_format) for targets in batches),
        return_exceptions=True,
    )
paper_tg_reasoning = [get_reasoning(r) for r in paper_tg_responses]

# Collect records; keep only those for the batch's own targets (or unmatched ones).
names = paper_samples_all.set_index("sample_id")
name_counts = paper_samples_all.name.str.strip().str.lower().value_counts()
rows, seen_unmatched = [], set()
for b, (targets, response) in enumerate(zip(batches, paper_tg_responses), 1):
    print(f"\n=== Request {b}/{len(batches)} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    if note:
        print(note)
    for r in (parsed or {}).get("properties", []):
        sid = r.get("sample_id")
        if sid is not None and sid not in targets:
            continue  # belongs to another batch's targets
        if sid is None:
            key = (r.get("paper_label"), r.get("value_raw"), r.get("method"))
            if key in seen_unmatched:
                continue
            seen_unmatched.add(key)
        problems = [p for p in [check_tg_record(r)] if p]
        if (r.get("match_basis") == "none") != (sid is None):
            problems.append("match_basis none must go with sample_id null")
        name = names["name"].get(sid, "") if sid is not None else ""
        unique_name = sid is not None and name_counts.get(name.strip().lower(), 0) == 1
        basis = r.get("match_basis")
        kept = sid is not None and (basis in STRONG_MATCHES or (basis == "name_only" and unique_name))
        rows.append({
            "kept": kept,
            "sample_id": sid,
            "sample": f"{name} | {names['identifier'].get(sid, '')}" if sid is not None else "",
            "paper_label": r.get("paper_label"),
            "identifying_data": r.get("identifying_data"),
            "match_basis": basis,
            "value_raw": r.get("value_raw"),
            "value": r.get("value"),
            "value_type": r.get("value_type"),
            "method": r.get("method"),
            "method_subtype": r.get("method_subtype"),
            "conditions": format_conditions(r.get("conditions")),
            "origin": r.get("origin"),
            "description": r.get("description"),
            "check": "; ".join(problems),
        })

paper_tg_records = pd.DataFrame(rows)
if paper_tg_records.empty:
    print("\nNo Tg values returned for this paper.")
else:
    # Same value assigned to more than one sample.
    kept_rows = paper_tg_records[paper_tg_records.kept]
    dup = kept_rows.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    idx = kept_rows.index[dup.to_numpy()]
    paper_tg_records.loc[idx, "check"] = (paper_tg_records.loc[idx, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nAll Tg values ({len(paper_tg_records)}; kept {int(paper_tg_records.kept.sum())}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(paper_tg_records)

    summary = paper_samples_all[["sample_id", "name", "identifier"]].copy()
    kept_values = (paper_tg_records[paper_tg_records.kept]
                   .groupby("sample_id").value_raw.apply(lambda v: "; ".join(v)))
    summary["kept_tg"] = summary.sample_id.map(kept_values).fillna("")
    print("\nKept Tg per sample:")
    with pd.option_context("display.max_rows", None):
        display(summary)

ID000588793: Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers
13 samples, 1 request(s)

=== Request 1/1 ===
--- Reasoning ---
The user wants to extract all glass transition temperature (Tg) values from the provided paper and link them to the provided sample list.

Let's scan the paper for mentions of Tg or glass transition temperature.
1. "Due to the rapid crystallization of PEB, its glass transition (Tg) was not detected by DSC (Fig. 2 b) [41]; By magnifying the DSC curve inFig. 2 b over the range of −20 to 20 °C (Fig. S5), a weak glass transition corresponding to the PEB segment becomes evident (−4 to 2 °C)."
   - This refers to the PEB segment Tg. It says "−4 to 2 °C".
   - It applies to the mBCPs in general, or specifically the ones tested. The text says "Due to the rapid crystallization of PEB, its glass transition (Tg) was not detected by DSC... a weak gl

,kept,sample_id,sample,paper_label,identifying_data,match_basis,value_raw,value,value_type,method,method_subtype,conditions,origin,description,check
0,False,None,,PEB segment,,none,−4 to 2 °C,2,range,DSC,unspecified,,measured,"Glass transition temperature of the PEB segment, observed as a weak transition in the DSC curve between −20 and 20 °C on the second heating scan.",
1,False,None,,PTMO segment,,none,−40 to −50 °C,-40,range,DMA,tan_delta_peak,,measured,"Glass transition temperature of the PTMO segment, identified from the relaxation peaks in the loss modulus and tan δ curves measured by DMA.",



Kept Tg per sample:


,sample_id,name,identifier,kept_tg
0,312469,(PEB-b-PTMO-b-PEB)n mBCP,P1-55K,
1,312470,(PEB-b-PTMO-b-PEB)n mBCP,P1-80K,
2,312471,(PEB-b-PTMO-b-PEB)n mBCP,P1-94K,
3,312472,(PEB-b-PTMO-b-PEB)n mBCP,P1-122K,
4,312473,(PEB-b-PTMO-b-PEB)n mBCP,P2-89K,
5,312474,(PEB-b-PTMO-b-PEB)n mBCP,P3-96K,
6,312475,(PEB-b-PTMO-b-PEB)n mBCP,P4-96K,
7,312476,P1-94K melt-spun fiber,P1-94K-OF,
8,312477,P1-94K post-drawn fiber,P1-94K-SF-2,
9,312478,P1-94K post-drawn fiber,P1-94K-SF-3,


## Send prompt 3 (per sample, other samples as context)

One request per sample in `run_ids`. Records are in `ctx_tg_records`; reasoning in `ctx_tg_reasoning`.

In [23]:
# Per-sample Tg extraction, with the paper's other samples listed for identification only.
ctx_samples = paper_samples
ctx_markdown = paper_markdown


def build_ctx_messages(sample_id):
    target = ctx_samples.loc[ctx_samples.sample_id == sample_id]
    if len(target) != 1:
        raise ValueError(f"Sample {sample_id} does not belong to paper {PAPER_UID}.")
    t = target.iloc[0]
    others = ctx_samples.loc[ctx_samples.sample_id != sample_id]
    values = {
        "paper_uid": PAPER_UID,
        "sample_id": str(sample_id),
        "sample_record": json.dumps({k: t[k] for k in ("name", "identifier", "aliases", "description")},
                                    ensure_ascii=False),
        "other_samples": "\n".join(sample_json(r) for r in others.itertuples()) or "[none]",
        "markdown_full_text": ctx_markdown,
    }
    user = re.sub(r"\{(paper_uid|sample_id|sample_record|other_samples|markdown_full_text)\}",
                  lambda m: values[m.group(1)], TG_CTX_USER_PROMPT)
    return [{"role": "system", "content": TG_CTX_SYSTEM_PROMPT}, {"role": "user", "content": user}]


ctx_ids = run_ids
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(ctx_samples)} samples in paper; running {len(ctx_ids)} request(s)")

# Messages are built here, from the current prompt cell, so prompt edits always take effect.
semaphore = asyncio.Semaphore(MAX_CONCURRENT)
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    async def run_one(sample_id):
        async with semaphore:
            return await ask_qwen(client, build_ctx_messages(sample_id), TG_RESPONSE_FORMAT)
    ctx_responses = dict(zip(ctx_ids, await asyncio.gather(*(run_one(i) for i in ctx_ids),
                                                             return_exceptions=True)))
ctx_tg_reasoning = {i: get_reasoning(r) for i, r in ctx_responses.items()}

names = ctx_samples.set_index("sample_id")
rows = []
for sample_id, response in ctx_responses.items():
    print(f"\n=== {sample_id}: {names['name'][sample_id]} | {names['identifier'][sample_id]} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    if records is None:
        print(note or "[No parsable answer]")
        continue
    if not records:
        print("No values reported for this sample.")
    for r in records:
        rows.append({
            "sample_id": sample_id,
            "sample": f"{names['name'][sample_id]} | {names['identifier'][sample_id]}",
            "method": r.get("method"),
            "method_subtype": r.get("method_subtype"),
            "value_raw": r.get("value_raw"),
            "value": r.get("value"),
            "value_type": r.get("value_type"),
            "unit": r.get("unit"),
            "origin": r.get("origin"),
            "conditions": format_conditions(r.get("conditions")),
            "description": r.get("description"),
            "check": check_tg_record(r),
        })
    if note:
        print(note)

ctx_tg_records = pd.DataFrame(rows)
if not ctx_tg_records.empty:
    dup = ctx_tg_records.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    ctx_tg_records.loc[dup, "check"] = (ctx_tg_records.loc[dup, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nTg values ({len(ctx_tg_records)}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(ctx_tg_records)
if RUN_ALL_SAMPLES:
    summary = ctx_samples[["sample_id", "name", "identifier"]].copy()
    values = (ctx_tg_records.groupby("sample_id").value_raw.apply(lambda v: "; ".join(v))
              if not ctx_tg_records.empty else pd.Series(dtype=str))
    summary["tg"] = summary.sample_id.map(values).fillna("")
    print("\nTg per sample:")
    with pd.option_context("display.max_rows", None):
        display(summary)

ID000588793: Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers
13 samples in paper; running 13 request(s)

=== 312469: (PEB-b-PTMO-b-PEB)n mBCP | P1-55K ===
--- Reasoning ---
The user wants to extract the glass transition temperature (Tg) for the sample "P1-55K" (sample_id=312469).
The paper discusses a series of mBCPs: P1-55K, P1-80K, P1-94K, P1-122K, P2-89K, P3-96K, P4-96K, and their fibers.
I need to find Tg values specifically for P1-55K.
Let's scan the text for Tg values.
Text says:
"Due to the rapid crystallization of PEB, its glass transition (Tg) was not detected by DSC (Fig. 2 b) [41]; By magnifying the DSC curve in Fig. 2 b over the range of −20 to 20 °C (Fig. S5), a weak glass transition corresponding to the PEB segment becomes evident (−4 to 2 °C)."
"However, no discernible transition corresponding to the Tg of the PTMO segment was detected in the DS

,sample_id,sample,method,method_subtype,value_raw,value,value_type,unit,origin,conditions,description,check
0,312469,(PEB-b-PTMO-b-PEB)n mBCP | P1-55K,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,heating_rate=10 °C/min; scan=second_heating; atmosphere=N2,"Tg of the PEB hard segment, observed as a weak transition in the second DSC heating scan recorded at 10 °C/min under nitrogen; the transition was identified by magnifying the curve between −20 and 20 °C.",assigned to several samples
1,312469,(PEB-b-PTMO-b-PEB)n mBCP | P1-55K,DMA,unspecified,−40 to −50 °C,-50,range,°C,measured,heating_rate=3 °C/min; frequency=1 Hz,"Tg of the PTMO soft segment, identified from relaxation peaks in the loss modulus and tan δ curves measured by DMA in tensile mode at 1 Hz with a heating rate of 3 °C/min over a temperature range of −100 to 60 °C.",assigned to several samples
2,312470,(PEB-b-PTMO-b-PEB)n mBCP | P1-80K,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,heating_rate=10 °C/min; scan=second_heating; atmosphere=N2,"Tg of the PEB segment, observed as a weak transition in the DSC curve (magnified between −20 and 20 °C) during the second heating scan at 10 °C/min under nitrogen.",assigned to several samples
3,312470,(PEB-b-PTMO-b-PEB)n mBCP | P1-80K,DMA,tan_delta_peak,−40 to −50 °C,-40,range,°C,measured,heating_rate=3 °C/min; frequency=1 Hz,"Tg of the PTMO segment, identified from relaxation peaks in the loss modulus and tan δ curves measured by DMA at 1 Hz with a heating rate of 3 °C/min.",assigned to several samples
4,312471,(PEB-b-PTMO-b-PEB)n mBCP | P1-94K,DSC,unspecified,−4 to 2 °C,2,range,°C,measured,"heating_rate=10 °C/min; scan=second_heating; atmosphere=N2; thermal_history=heated to 100 °C, held for 2 min, then cooled to −20 °C at 10 °C/min","Glass transition temperature of the PEB hard segment, observed as a weak transition in the second DSC heating scan recorded at 10 °C/min under nitrogen after removing thermal history by heating to 100 °C, holding for 2 min, and cooling to −20 °C at 10 °C/min.",
5,312471,(PEB-b-PTMO-b-PEB)n mBCP | P1-94K,DMA,tan_delta_peak,−40 to −50 °C,-40,range,°C,measured,frequency=1 Hz; heating_rate=3 °C/min; temperature_range=-100 to 60 °C (range),"Glass transition temperature of the PTMO soft segment, identified as the tan δ peak in DMA measurements performed at 1 Hz with a heating rate of 3 °C/min over a temperature range of −100 to 60 °C.",



Tg per sample:


,sample_id,name,identifier,tg
0,312469,(PEB-b-PTMO-b-PEB)n mBCP,P1-55K,−4 to 2 °C; −40 to −50 °C
1,312470,(PEB-b-PTMO-b-PEB)n mBCP,P1-80K,−4 to 2 °C; −40 to −50 °C
2,312471,(PEB-b-PTMO-b-PEB)n mBCP,P1-94K,−4 to 2 °C; −40 to −50 °C
3,312472,(PEB-b-PTMO-b-PEB)n mBCP,P1-122K,
4,312473,(PEB-b-PTMO-b-PEB)n mBCP,P2-89K,
5,312474,(PEB-b-PTMO-b-PEB)n mBCP,P3-96K,
6,312475,(PEB-b-PTMO-b-PEB)n mBCP,P4-96K,
7,312476,P1-94K melt-spun fiber,P1-94K-OF,
8,312477,P1-94K post-drawn fiber,P1-94K-SF-2,
9,312478,P1-94K post-drawn fiber,P1-94K-SF-3,


## Compare the three prompts

One row per sample that prompts 1 and 3 were run for. `p2_kept` are the values prompt 2 kept for that sample; `p2_weak` are values prompt 2 linked to the sample but did not keep (`name_only` with a shared name). `same` is True when all three prompts return the same set of values (by `value_raw` and method). Values prompt 2 linked to no sample are listed below the table.

In [24]:
def raw_labels(df, mask=None):
    # Labels per sample_id, built from the flattened record columns.
    if df is None or df.empty:
        return {}
    if mask is not None:
        df = df[mask]
    out = {}
    for r in df.to_dict("records"):
        method = r.get("method") or ""
        if r.get("method_subtype") not in (None, "unspecified"):
            method += f"/{r['method_subtype']}"
        cond = r.get("conditions") or ""
        out.setdefault(r["sample_id"], []).append(f"{r.get('value_raw')} [{method}{'; ' + cond if cond else ''}]")
    return out


def value_keys(df, sample_id, mask=None):
    if df is None or df.empty:
        return set()
    d = df if mask is None else df[mask]
    d = d[d.sample_id == sample_id]
    return {(str(v).strip(), m) for v, m in zip(d.value_raw, d.method)}


p2 = paper_tg_records if "paper_tg_records" in globals() else pd.DataFrame()
p1 = p1_records if "p1_records" in globals() else pd.DataFrame()
p3 = ctx_tg_records if "ctx_tg_records" in globals() else pd.DataFrame()

p1_map = raw_labels(p1)
p3_map = raw_labels(p3)
p2_kept_map = raw_labels(p2, p2.kept if not p2.empty else None)
p2_weak_map = raw_labels(p2, (~p2.kept & p2.sample_id.notna()) if not p2.empty else None)

rows = []
for sample_id in run_ids:
    r = names.loc[sample_id]
    k1 = value_keys(p1, sample_id)
    k2 = value_keys(p2, sample_id, p2.kept if not p2.empty else None)
    k3 = value_keys(p3, sample_id)
    rows.append({
        "sample_id": sample_id,
        "sample": f"{r['name']} | {r['identifier']}",
        "p1_per_sample": " | ".join(p1_map.get(sample_id, [])),
        "p2_kept": " | ".join(p2_kept_map.get(sample_id, [])),
        "p2_weak": " | ".join(p2_weak_map.get(sample_id, [])),
        "p3_with_context": " | ".join(p3_map.get(sample_id, [])),
        "same": k1 == k2 == k3,
    })
comparison = pd.DataFrame(rows)
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"Samples compared: {len(comparison)}; all three agree on {int(comparison.same.sum())}")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(comparison)

if not p2.empty:
    unmatched = p2[p2.sample_id.isna()]
    print(f"\nPrompt 2 values linked to no sample ({len(unmatched)}):")
    if len(unmatched):
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(unmatched[["paper_label", "identifying_data", "value_raw", "method", "conditions"]])
    others = p2[p2.kept & ~p2.sample_id.isin(run_ids)]
    if len(others) and not RUN_ALL_SAMPLES:
        print(f"\nPrompt 2 also kept {len(others)} value(s) for samples not run with prompts 1 and 3 "
              f"(set RUN_ALL_SAMPLES = True to compare them).")

ID000588793: Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers
Samples compared: 13; all three agree on 8


,sample_id,sample,p1_per_sample,p2_kept,p2_weak,p3_with_context,same
0,312469,(PEB-b-PTMO-b-PEB)n mBCP | P1-55K,,,,−4 to 2 °C [DSC; heating_rate=10 °C/min; scan=second_heating; atmosphere=N2] | −40 to −50 °C [DMA; heating_rate=3 °C/min; frequency=1 Hz],False
1,312470,(PEB-b-PTMO-b-PEB)n mBCP | P1-80K,−4 to 2 °C [DSC; heating_rate=10 °C/min; scan=second_heating; atmosphere=N2] | −40 to −50 °C [DMA/tan_delta_peak; frequency=1 Hz; heating_rate=3 °C/min],,,−4 to 2 °C [DSC; heating_rate=10 °C/min; scan=second_heating; atmosphere=N2] | −40 to −50 °C [DMA/tan_delta_peak; heating_rate=3 °C/min; frequency=1 Hz],False
2,312471,(PEB-b-PTMO-b-PEB)n mBCP | P1-94K,−4 to 2 °C [DSC; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; temperature_range=-20 to 100 °C (range); atmosphere=N2; specimen_form=film] | −40 to −50 °C [DMA/tan_delta_peak; heating_rate=3 °C/min; frequency=1 Hz; temperature_range=-100 to 60 °C (range); specimen_form=dumbbell-shaped film],,,"−4 to 2 °C [DSC; heating_rate=10 °C/min; scan=second_heating; atmosphere=N2; thermal_history=heated to 100 °C, held for 2 min, then cooled to −20 °C at 10 °C/min] | −40 to −50 °C [DMA/tan_delta_peak; frequency=1 Hz; heating_rate=3 °C/min; temperature_range=-100 to 60 °C (range)]",False
3,312472,(PEB-b-PTMO-b-PEB)n mBCP | P1-122K,,,,,True
4,312473,(PEB-b-PTMO-b-PEB)n mBCP | P2-89K,,,,,True
5,312474,(PEB-b-PTMO-b-PEB)n mBCP | P3-96K,−40 to −50 °C [DMA/tan_delta_peak; frequency=1 Hz; heating_rate=3 °C/min; temperature_range=60 °C (range)] | −4 to 2 °C [DSC; heating_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=100 °C (range)],,,,False
6,312475,(PEB-b-PTMO-b-PEB)n mBCP | P4-96K,−4 to 2 °C [DSC; heating_rate=10 °C/min; scan=second_heating; atmosphere=N2] | −40 to −50 °C [DMA/tan_delta_peak; frequency=1 Hz; heating_rate=3 °C/min; temperature_range=-100 to 60 °C; specimen_form=dumbbell-shaped sample],,,,False
7,312476,P1-94K melt-spun fiber | P1-94K-OF,,,,,True
8,312477,P1-94K post-drawn fiber | P1-94K-SF-2,,,,,True
9,312478,P1-94K post-drawn fiber | P1-94K-SF-3,,,,,True



Prompt 2 values linked to no sample (2):


,paper_label,identifying_data,value_raw,method,conditions
0,PEB segment,,−4 to 2 °C,DSC,
1,PTMO segment,,−40 to −50 °C,DMA,


## All records with all fields

Prints every record each prompt returned, as JSON with all fields, grouped by sample (prompts 1 and 3) or by request (prompt 2). For prompt 2 each record also shows which sample it was linked to and whether code kept it. Run after the send cells.

In [25]:
# Print every property record from the three prompts with all fields, as returned by the model.
# Uses the responses from the send cells above; run those first.

def print_records(title, records, extra=None):
    print(f"\n--- {title}: {len(records)} record(s) ---")
    if not records:
        print("(no records)")
    for n, r in enumerate(records, 1):
        print(f"\n[{n}]" + (f" {extra(r)}" if extra else ""))
        print(json.dumps(r, ensure_ascii=False, indent=2))


def sample_title(sample_id):
    r = names.loc[sample_id]
    return f"{sample_id}: {r['name']} | {r['identifier']}"


def records_of(response):
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    return records, note


print("=" * 80)
print(f"PROMPT 1 (per sample) - {PAPER_UID}")
print("=" * 80)
if "p1_responses" not in globals():
    print("Not run.")
else:
    for sample_id, response in p1_responses.items():
        records, note = records_of(response)
        print_records(sample_title(sample_id), records or [])
        if records is None or note:
            print(note or "[No parsable answer]")

print("\n" + "=" * 80)
print(f"PROMPT 2 (per paper) - {PAPER_UID}")
print("=" * 80)
if "paper_tg_responses" not in globals():
    print("Not run.")
else:
    shared_names = paper_samples.name.str.strip().str.lower().value_counts()
    def p2_status(r):
        sid = r.get("sample_id")
        if sid is None:
            return "-> no sample (not kept)"
        name = names["name"].get(sid, "")
        unique = shared_names.get(name.strip().lower(), 0) == 1
        kept = r.get("match_basis") in STRONG_MATCHES or (r.get("match_basis") == "name_only" and unique)
        return f"-> {sample_title(sid)} ({'kept' if kept else 'not kept'})"
    for b, (targets, response) in enumerate(zip(batches, paper_tg_responses), 1):
        records, note = records_of(response)
        print_records(f"Request {b}/{len(batches)} (targets: {', '.join(map(str, targets))})",
                      records or [], extra=p2_status)
        if records is None or note:
            print(note or "[No parsable answer]")

print("\n" + "=" * 80)
print(f"PROMPT 3 (per sample, other samples as context) - {PAPER_UID}")
print("=" * 80)
if "ctx_responses" not in globals():
    print("Not run.")
else:
    for sample_id, response in ctx_responses.items():
        records, note = records_of(response)
        print_records(sample_title(sample_id), records or [])
        if records is None or note:
            print(note or "[No parsable answer]")

PROMPT 1 (per sample) - ID000588793

--- 312469: (PEB-b-PTMO-b-PEB)n mBCP | P1-55K: 0 record(s) ---
(no records)

--- 312470: (PEB-b-PTMO-b-PEB)n mBCP | P1-80K: 2 record(s) ---

[1]
{
  "name": "glass_transition_temperature",
  "method": "DSC",
  "method_subtype": "unspecified",
  "value_raw": "−4 to 2 °C",
  "value": 2,
  "value_type": "range",
  "unit": "°C",
  "conditions": [
    {
      "name": "heating_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "scan",
      "type": "exact",
      "value": "second_heating",
      "unit": null,
      "description": null
    },
    {
      "name": "atmosphere",
      "type": "exact",
      "value": "N2",
      "unit": null,
      "description": null
    }
  ],
  "origin": "measured",
  "description": "Glass transition temperature of the PEB hard segment, observed as a weak transition in the second DSC heating scan recorded at 10 °C/min under nitrogen atmosphere."
}

[